01 — Setup

In [1]:
import pandas as pd

from src.config.settings import Settings
from src.canonical.nact_transformer import NACTTransformer
from src.canonical.pipeline import CanonicalPipeline

pipeline = CanonicalPipeline()

landing_metadata_df = pd.read_csv(Settings.LANDING_METADATA_PATH)

26/06/26 20:12:11 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).


02 — Buscar NACT EY automaticamente

In [2]:
nact_ey_df = landing_metadata_df[
    (landing_metadata_df["relative_path"].str.contains("NACT", na=False))
    & (landing_metadata_df["file_extension"] == ".xlsb")
]

nact_ey_row = (
    nact_ey_df
    .sort_values("snapshot_date")
    .iloc[-1]
    .to_dict()
)

nact_ey_row

{'snapshot_date': '2024-02-01_0800',
 'source_file': '202402_ADMIN.xlsb',
 'source_path': '/app/data/raw_local/RAW/2024-02-01_0800/NACT/202402_ADMIN.xlsb',
 'relative_path': '2024-02-01_0800/NACT/202402_ADMIN.xlsb',
 'landing_bucket': 'contracts',
 'landing_key': 'landing/snapshot_date=2024-02-01_0800/NACT/202402_ADMIN.xlsb',
 'file_extension': '.xlsb',
 'file_size_bytes': 20510580,
 'file_hash': 'fd09ffbb01a905f97036fb472ac085932c4f15e53502d3d60e79aed4b8e3089e',
 'discovered_at': '2026-06-25T19:10:15.519455+00:00',
 'upload_status': 'SUCCESS',
 'uploaded_at': '2026-06-25T19:10:18.555554+00:00',
 'error_message': nan}

03 — Testar transformer EY

In [3]:
ey_raw_df = pd.read_excel(
    nact_ey_row["source_path"],
    sheet_name="RACT",
    engine="pyxlsb",
    header=None,
    dtype=str,
)

ey_canonical_df = NACTTransformer.transform_ey_ract(
    raw_df=ey_raw_df,
    snapshot_date=nact_ey_row["snapshot_date"],
)

print(ey_canonical_df.shape)
ey_canonical_df.head()

(4518, 15)


,nact_internal_id,contract_number_raw,contract_number,supplier_cnpj,supplier_name,operation_location,supplier_role,competence,compliance_score,pending_issues_summary,source_vendor,source_layout,canonical_entity,snapshot_date,processed_at
0,4900,ALPA5900065869,5900065869,00.973.749/0016-00,TOP SERVICE SERVICOS E SISTEMAS LTDA.,PA/MARABÁ,None,None,None,None,EY,ey_ract_v1,nact_contracts,2024-02-01_0800,2026-06-26T20:12:25.813820+00:00
1,4902,BAOVALE5900052656,5900052656,02.035.105/0001-01,UNIMAR TRANSPORTES LTDA,ES/VITÓRIA,None,None,None,None,EY,ey_ract_v1,nact_contracts,2024-02-01_0800,2026-06-26T20:12:25.813820+00:00
2,4927,CPBS5900030295,5900030295,22.320.881/0001-60,Tradimaq Ltda.,RJ/RIO DE JANEIRO,None,None,None,None,EY,ey_ract_v1,nact_contracts,2024-02-01_0800,2026-06-26T20:12:25.813820+00:00
3,4931,CPBS5900055532,5900055532,52.548.435/0001-79,JSL S/A,RJ/ITAGUAÍ (CPBS),None,None,None,None,EY,ey_ract_v1,nact_contracts,2024-02-01_0800,2026-06-26T20:12:25.813820+00:00
4,4932,CPBS5900059303,5900059303,07.147.444/0001-01,REFRAMAX ENGENHARIA S/A,RJ/ITAGUAÍ (CPBS),None,None,None,None,EY,ey_ract_v1,nact_contracts,2024-02-01_0800,2026-06-26T20:12:25.813820+00:00


04 — Executar pipeline EY

In [4]:
ey_output_path = pipeline.run_nact_contracts(
    source_path=nact_ey_row["source_path"],
    snapshot_date=nact_ey_row["snapshot_date"],
)

print(ey_output_path)

ey_spark_df = pipeline.spark.read.parquet(ey_output_path)

ey_spark_df.printSchema()
ey_spark_df.show(5)

26/06/26 20:12:38 WARN MetricsConfig: Cannot locate configuration: tried hadoop-metrics2-s3a-file-system.properties,hadoop-metrics2.properties


s3a://contracts/canonical/entity=nact_contracts/snapshot_date=2024-02-01_0800/
root
 |-- nact_internal_id: string (nullable = true)
 |-- contract_number_raw: string (nullable = true)
 |-- contract_number: string (nullable = true)
 |-- supplier_cnpj: string (nullable = true)
 |-- supplier_name: string (nullable = true)
 |-- operation_location: string (nullable = true)
 |-- supplier_role: string (nullable = true)
 |-- competence: string (nullable = true)
 |-- compliance_score: string (nullable = true)
 |-- pending_issues_summary: string (nullable = true)
 |-- source_vendor: string (nullable = true)
 |-- source_layout: string (nullable = true)
 |-- canonical_entity: string (nullable = true)
 |-- snapshot_date: string (nullable = true)
 |-- processed_at: string (nullable = true)

+----------------+-------------------+---------------+------------------+--------------------+--------------------+-------------+----------+----------------+----------------------+-------------+-------------+-----

05 — Buscar NACT Deloitte automaticamente

In [5]:
nact_deloitte_df = landing_metadata_df[
    (landing_metadata_df["relative_path"].str.contains("NACT", na=False))
    & (landing_metadata_df["file_extension"] == ".xlsx")
]

nact_deloitte_row = (
    nact_deloitte_df
    .sort_values("snapshot_date")
    .iloc[-1]
    .to_dict()
)

nact_deloitte_row

{'snapshot_date': '2024-02-01_0800',
 'source_file': '2024-02-Monitoramento_Mensal_Consolidado.xlsx',
 'source_path': '/app/data/raw_local/RAW/2024-02-01_0800/NACT/2024-02-Monitoramento_Mensal_Consolidado.xlsx',
 'relative_path': '2024-02-01_0800/NACT/2024-02-Monitoramento_Mensal_Consolidado.xlsx',
 'landing_bucket': 'contracts',
 'landing_key': 'landing/snapshot_date=2024-02-01_0800/NACT/2024-02-Monitoramento_Mensal_Consolidado.xlsx',
 'file_extension': '.xlsx',
 'file_size_bytes': 1000268,
 'file_hash': '35e43f97cb9521e0d3c14cce0010d72009ad11f15421aca87c21ecf0b3cbf545',
 'discovered_at': '2026-06-25T19:10:14.998684+00:00',
 'upload_status': 'SUCCESS',
 'uploaded_at': '2026-06-25T19:10:17.898098+00:00',
 'error_message': nan}

06 — Testar transformer Deloitte

In [6]:
deloitte_raw_df = pd.read_excel(
    nact_deloitte_row["source_path"],
    sheet_name=0,
    header=None,
    dtype=str,
)

deloitte_canonical_df = NACTTransformer.transform_deloitte_contracts(
    raw_df=deloitte_raw_df,
    snapshot_date=nact_deloitte_row["snapshot_date"],
)

print(deloitte_canonical_df.shape)
deloitte_canonical_df.head()

(79, 15)


,nact_internal_id,contract_number_raw,contract_number,supplier_cnpj,supplier_name,operation_location,supplier_role,competence,compliance_score,pending_issues_summary,source_vendor,source_layout,canonical_entity,snapshot_date,processed_at
0,NaN,5900101005,5900101005,27.126.997/0001-87,FORTANKS INDUSTRIA DE TANQUES DE CONCRETO LTDA,VITÓRIA,CONTRATADA,Outubro/2023,56%,10 / 6,Deloitte,deloitte_contracts_v1,nact_contracts,2024-02-01_0800,2026-06-26T20:12:48.695693+00:00
1,NaN,5900101086,5900101086,30.677.132/0001-13,FORTES ENGENHARIA LTDA,VITÓRIA,CONTRATADA,Outubro/2023,93%,1 / 0,Deloitte,deloitte_contracts_v1,nact_contracts,2024-02-01_0800,2026-06-26T20:12:48.695693+00:00
2,NaN,5900101267,5900101267,77.591.402/0001-32,JOTA ELE CONSTRUCOES CIVIS S A,VITÓRIA,CONTRATADA,Outubro/2023,100%,0 / 0,Deloitte,deloitte_contracts_v1,nact_contracts,2024-02-01_0800,2026-06-26T20:12:48.695693+00:00
3,NaN,5900101350,5900101350,03.204.109/0001-39,CIABRASIL ENGENHARIA E SISTEMAS CERAMICOS LTDA,VITÓRIA,CONTRATADA,Outubro/2023,100%,0 / 0,Deloitte,deloitte_contracts_v1,nact_contracts,2024-02-01_0800,2026-06-26T20:12:48.695693+00:00
4,NaN,5900101350,5900101350,28.091.591/0001-79,SCAN SOLO SERVICOS LTDA,VITÓRIA,SUB-CONTRATADA,Outubro/2023,100%,0 / 0,Deloitte,deloitte_contracts_v1,nact_contracts,2024-02-01_0800,2026-06-26T20:12:48.695693+00:00


07 — Executar pipeline Deloitte

In [7]:
deloitte_output_path = pipeline.run_nact_contracts(
    source_path=nact_deloitte_row["source_path"],
    snapshot_date=nact_deloitte_row["snapshot_date"],
)

print(deloitte_output_path)

deloitte_spark_df = pipeline.spark.read.parquet(deloitte_output_path)

deloitte_spark_df.printSchema()
deloitte_spark_df.show(5)

s3a://contracts/canonical/entity=nact_contracts/snapshot_date=2024-02-01_0800/
root
 |-- nact_internal_id: string (nullable = true)
 |-- contract_number_raw: string (nullable = true)
 |-- contract_number: string (nullable = true)
 |-- supplier_cnpj: string (nullable = true)
 |-- supplier_name: string (nullable = true)
 |-- operation_location: string (nullable = true)
 |-- supplier_role: string (nullable = true)
 |-- competence: string (nullable = true)
 |-- compliance_score: string (nullable = true)
 |-- pending_issues_summary: string (nullable = true)
 |-- source_vendor: string (nullable = true)
 |-- source_layout: string (nullable = true)
 |-- canonical_entity: string (nullable = true)
 |-- snapshot_date: string (nullable = true)
 |-- processed_at: string (nullable = true)

+----------------+-------------------+---------------+------------------+--------------------+------------------+-------------+------------+----------------+----------------------+-------------+-------------------

08 — Summary

In [8]:
canonical_validation = {
    "ey_source_file": nact_ey_row["source_file"],
    "ey_snapshot_date": nact_ey_row["snapshot_date"],
    "ey_output_path": ey_output_path,
    "deloitte_source_file": nact_deloitte_row["source_file"],
    "deloitte_snapshot_date": nact_deloitte_row["snapshot_date"],
    "deloitte_output_path": deloitte_output_path,
    "status": "SUCCESS",
}

canonical_validation

{'ey_source_file': '202402_ADMIN.xlsb',
 'ey_snapshot_date': '2024-02-01_0800',
 'ey_output_path': 's3a://contracts/canonical/entity=nact_contracts/snapshot_date=2024-02-01_0800/',
 'deloitte_source_file': '2024-02-Monitoramento_Mensal_Consolidado.xlsx',
 'deloitte_snapshot_date': '2024-02-01_0800',
 'deloitte_output_path': 's3a://contracts/canonical/entity=nact_contracts/snapshot_date=2024-02-01_0800/',
 'status': 'SUCCESS'}

09 — Stop Spark

In [9]:
#pipeline.spark.stop()

In [10]:
import importlib
import src.canonical.pipeline

importlib.reload(src.canonical.pipeline)

from src.canonical.pipeline import CanonicalPipeline

pipeline = CanonicalPipeline()

In [11]:
landing_metadata_df = pd.read_csv(Settings.LANDING_METADATA_PATH)

In [13]:
landing_metadata_df.head()

,snapshot_date,source_file,source_path,relative_path,landing_bucket,landing_key,file_extension,file_size_bytes,file_hash,discovered_at,upload_status,uploaded_at,error_message
0,2022-11-01_0800,ControleMedicoesPagamentos.csv,/app/data/raw_local/RAW/2022-11-01_0800/CONTRO...,2022-11-01_0800/CONTROLE DE MEDICOES E PAGAMEN...,contracts,landing/snapshot_date=2022-11-01_0800/CONTROLE...,.csv,39634,b73d09f9dc6766eb918f3e1149649f758e0338d0f42f86...,2026-06-25T19:10:13.253927+00:00,SUCCESS,2026-06-25T19:10:15.734075+00:00,NaN
1,2022-11-01_0800,Exportação_bm_acompanhamento.xlsx,/app/data/raw_local/RAW/2022-11-01_0800/CONTRO...,2022-11-01_0800/CONTROLE DE MEDICOES EM ANDAME...,contracts,landing/snapshot_date=2022-11-01_0800/CONTROLE...,.xlsx,18912,84df6ac3cb58f7ca789bf83e192ad99369bb1d35c6b41f...,2026-06-25T19:10:13.277716+00:00,SUCCESS,2026-06-25T19:10:15.765103+00:00,NaN
2,2022-11-01_0800,2022-11-Monitoramento_Mensal_Consolidado.xlsx,/app/data/raw_local/RAW/2022-11-01_0800/NACT/2...,2022-11-01_0800/NACT/2022-11-Monitoramento_Men...,contracts,landing/snapshot_date=2022-11-01_0800/NACT/202...,.xlsx,736830,b88f5e71367a511272701888754604fed9fba30c07e070...,2026-06-25T19:10:13.321252+00:00,SUCCESS,2026-06-25T19:10:15.834471+00:00,NaN
3,2022-11-01_0800,202211_ADMIN.xlsb,/app/data/raw_local/RAW/2022-11-01_0800/NACT/2...,2022-11-01_0800/NACT/202211_ADMIN.xlsb,contracts,landing/snapshot_date=2022-11-01_0800/NACT/202...,.xlsb,20251932,5d05acde002b287d9068b5b4b8dc34dc04609d1cc16cbb...,2026-06-25T19:10:13.883625+00:00,SUCCESS,2026-06-25T19:10:16.509864+00:00,NaN
4,2022-11-01_0800,Pendências-010223.xlsx,/app/data/raw_local/RAW/2022-11-01_0800/PENDEN...,2022-11-01_0800/PENDENCIAS RDO/Pendências-0102...,contracts,landing/snapshot_date=2022-11-01_0800/PENDENCI...,.xlsx,23053,3cca4c48eaa038cee8bd806f6cae89ca1de1e6ee0e9996...,2026-06-25T19:10:13.917630+00:00,SUCCESS,2026-06-25T19:10:16.529224+00:00,NaN


## 08 — Buscar Analítico SGC automaticamente

In [14]:
sgc_contracts_df = landing_metadata_df[
    landing_metadata_df["source_file"] == "AnaliticoProjeto.csv"
]

sgc_contracts_row = (
    sgc_contracts_df
    .sort_values("snapshot_date")
    .iloc[-1]
    .to_dict()
)

sgc_contracts_row

{'snapshot_date': '2024-02-01_0800',
 'source_file': 'AnaliticoProjeto.csv',
 'source_path': '/app/data/raw_local/RAW/2024-02-01_0800/RELATORIO ANALITICO DO PROJETO/AnaliticoProjeto.csv',
 'relative_path': '2024-02-01_0800/RELATORIO ANALITICO DO PROJETO/AnaliticoProjeto.csv',
 'landing_bucket': 'contracts',
 'landing_key': 'landing/snapshot_date=2024-02-01_0800/RELATORIO ANALITICO DO PROJETO/AnaliticoProjeto.csv',
 'file_extension': '.csv',
 'file_size_bytes': 67337,
 'file_hash': '7105eae169d8b48a7691d29b708f5ad2e7a530922a7efdf56c084ba20255b6bd',
 'discovered_at': '2026-06-25T19:10:15.649762+00:00',
 'upload_status': 'SUCCESS',
 'uploaded_at': '2026-06-25T19:10:18.742074+00:00',
 'error_message': nan}

## 09 — Executar pipeline SGC Contracts

In [15]:
sgc_output_path = pipeline.run_sgc_contracts(
    source_path=sgc_contracts_row["source_path"],
    snapshot_date=sgc_contracts_row["snapshot_date"],
)

print(sgc_output_path)

sgc_spark_df = pipeline.spark.read.parquet(sgc_output_path)

sgc_spark_df.printSchema()
sgc_spark_df.show(5)

26/06/26 20:13:15 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.


s3a://contracts/canonical/entity=sgc_contracts/snapshot_date=2024-02-01_0800/
root
 |-- contract_number_raw: string (nullable = true)
 |-- contract_number: string (nullable = true)
 |-- supplier_name: string (nullable = true)
 |-- contract_description: string (nullable = true)
 |-- project_area: string (nullable = true)
 |-- currency: string (nullable = true)
 |-- total_value: string (nullable = true)
 |-- po_value: string (nullable = true)
 |-- total_tacs: string (nullable = true)
 |-- total_claims: string (nullable = true)
 |-- total_adjustments: string (nullable = true)
 |-- total_advance: string (nullable = true)
 |-- measured_value: string (nullable = true)
 |-- remaining_to_measure: string (nullable = true)
 |-- contractual_block: string (nullable = true)
 |-- remaining_to_pay: string (nullable = true)
 |-- remaining_percentage: string (nullable = true)
 |-- start_date: string (nullable = true)
 |-- end_date: string (nullable = true)
 |-- remaining_days: string (nullable = true)
